In [ ]:
# Dự đoán điểm thi sinh viên (Hồi quy tuyến tính)


In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

MODEL_PATH = Path("artifacts/linear_regression.joblib")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

artifact = joblib.load(MODEL_PATH)
model = artifact["model"]
feature_names = artifact["feature_names"]

In [ ]:
hours = widgets.FloatSlider(value=6, min=0, max=20, step=0.5, description="Giờ học:", continuous_update=False)
attendance = widgets.FloatSlider(value=80, min=50, max=100, step=1, description="Chuyên cần:", continuous_update=False)
assignments = widgets.IntSlider(value=7, min=0, max=10, step=1, description="Bài tập:", continuous_update=False)
previous = widgets.FloatSlider(value=6.5, min=0, max=10, step=0.1, description="Điểm trước:", continuous_update=False)

predict_button = widgets.Button(description="Dự đoán", button_style="primary")
output = widgets.Output()

def input_frame():
    values = {
        "hours_studied": hours.value,
        "attendance": attendance.value,
        "assignments_completed": assignments.value,
        "previous_score": previous.value,
    }
    return pd.DataFrame([[values[name] for name in feature_names]], columns=feature_names)

def on_predict(_):
    predict_button.disabled = True
    with output:
        output.clear_output()
        try:
            raw = float(model.predict(input_frame())[0])
            val = np.clip(raw, 0, 10)
            print(f"Điểm thi dự đoán: {val:.2f} / 10")
        except Exception:
            print("Không thể dự đoán. Hãy kiểm tra dữ liệu và artifact.")
        finally:
            predict_button.disabled = False

predict_button.on_click(on_predict)

# Tạo tiêu đề và mô tả trực quan
title_widget = widgets.HTML(value="<h1 style='font-size: 24px; font-weight: bold; margin-bottom: 4px;'>Dự đoán điểm thi</h1>")
desc_widget = widgets.HTML(value="<p style='color: #555; margin-bottom: 20px; font-size: 14px;'>Nhập các thông tin hợp lệ, sau đó nhấn <b>Dự đoán</b>. Mô hình minh họa chỉ dùng cho mục đích giảng dạy.</p>")

# Gom nhóm và bố trí giao diện đẹp mắt
form_box = widgets.VBox([
    title_widget,
    desc_widget,
    hours, attendance, assignments, previous, 
    widgets.HBox([predict_button], layout=widgets.Layout(margin='15px 0px 10px 0px')),
    output
], layout=widgets.Layout(padding='24px', border='1px solid #e5e7eb', border_radius='12px', max_width='600px', background_color='#ffffff'))

display(form_box)